# 欢迎来到第 2 天实验！


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">开始之前——</h2>
            <span style="color:#f71;">我想先指向课程的实用资源页，其中包含全部幻灯片链接。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            请收藏此页，我会持续补充更多有用链接。
            </span>
        </td>
    </tr>
</table>

## 首先——聊聊 Chat Completions API

1. 调用 LLM 最简单的方式
2. 之所以叫 Chat Completions，是因为它在说：「这是一段对话，请预测接下来该说什么」
3. Chat Completions API 由 OpenAI 发明，但因太流行几乎人人都在用！

### 我们先再次调用 OpenAI——不用担心非 OpenAI 用户，马上轮到你们！


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not gemini_api_key:
    print("No Gemini API key found. Please check your .env file.")
else:
    print("Gemini API key found!")

## 你知道什么是 Endpoint（端点）吗？

若不清楚，请复习 guides 文件夹里的 Technical Foundations 指南

另外，这里有一个你可能感兴趣的端点……

In [ ]:
import requests

headers = {"Authorization": f"Bearer {gemini_api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gemini-2.5-flash",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

In [ ]:
response = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()["choices"][0]["message"]["content"]

# openai 包是什么？

它是一个 Python 客户端库。

本质上只是对这个 HTTP 端点调用的一层封装。

让你用干净的 Python 代码，而不用折腾难看的 JSON 对象。

仅此而已。开源且轻量。有人以为它内含 OpenAI 模型代码——其实没有！


In [ ]:
# 创建 OpenAI 客户端

from openai import OpenAI

gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
response = gemini.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "Tell me a fun fact"}
    ]
)

response.choices[0].message.content

## 然后发生了很棒的事：

OpenAI 的 Chat Completions API 太受欢迎，其他模型厂商也做了相同形态的端点。

它们被称为「OpenAI Compatible Endpoints」（OpenAI 兼容端点）。

例如 Google 做了一个： https://generativelanguage.googleapis.com/v1beta/openai/

OpenAI 也很慷慨：你可以直接用他们为 GPT 做的同一个客户端库，只需指定不同的端点 URL 与 Key，就能调用其他厂商。

例如你可以这样写：

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

说清楚：代码里虽有 OpenAI，但我们只用这个轻量 Python 客户端去打端点——这里并不涉及 OpenAI 模型。

若仍困惑，请复习 Guides 文件夹中的 Guide 9！

现在来试试！

## 这部分可选——若想试用 Google Gemini，请访问：

https://aistudio.google.com/

并在此创建 API Key：

https://aistudio.google.com/api-keys

然后把 Key 写入 `.env`，改完后务必保存：

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## Ollama 也提供 OpenAI 兼容端点

……而且跑在你自己的机器上！

若下一格没有打印 "Ollama is running"，请打开终端运行 `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### 从 Meta 下载 llama3.2

若电脑配置较低，可改成 llama3.2:1b。

不要用 llama3.3 或 llama4！对你的电脑来说通常太大……

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

# 家庭作业练习

把第 1 天的网页摘要项目升级为：通过 Ollama 在本地跑开源模型，而不是 OpenAI

若不想用付费 API，后续项目也都能用这套技巧。

**优点：**
1. 无 API 费用——开源
2. 数据不离开本机

**缺点：**
1. 能力明显弱于前沿模型

## 回顾 Ollama 安装

访问 [ollama.com](https://ollama.com) 安装即可！

安装完成后，ollama 服务通常已在本地运行。  
若你访问：  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [ ]:
from openai import OpenAI
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from IPython.display import Markdown
import time
MODEL = "llama3.2"

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
options = Options()
driver = webdriver.Chrome(options=options)

driver.get("https://www.sephora.com")
time.sleep(5)
html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

for tag in soup(["script", "style"]):
    tag.decompose()

text = soup.get_text(separator="\n", strip=True)

driver.quit()
system_prompt = """
You are an arrogant beauty advisor.
Summarize the Sephora homepage and recommend interesting products.
Return the answer in markdown.
"""

user_prompt = text[:12000]
response = ollama.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)
Markdown(response.choices[0].message.content)